<a href="https://colab.research.google.com/github/arelkeselbri/gsi073/blob/main/Aula02_tokenizacao_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 - Tokenização

## Parte 1 - Pré-tokenização


In [29]:
from tokenizers import Tokenizer, models, pre_tokenizers

tok_ws = Tokenizer(models.BPE())
tok_ws.pre_tokenizer = pre_tokenizers.Whitespace()
frase = "Não, será punido o criminoso."

print(tok_ws.pre_tokenizer.pre_tokenize_str(frase))


[('Não', (0, 3)), (',', (3, 4)), ('será', (5, 9)), ('punido', (10, 16)), ('o', (17, 18)), ('criminoso', (19, 28)), ('.', (28, 29))]


## *Punctuation + Whitespace*

In [30]:
tok_punc = Tokenizer(models.BPE())
tok_punc.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation()
])
print(tok_punc.pre_tokenizer.pre_tokenize_str(frase))

[('Não', (0, 3)), (',', (3, 4)), ('será', (5, 9)), ('punido', (10, 16)), ('o', (17, 18)), ('criminoso', (19, 28)), ('.', (28, 29))]


**Pretokenizer: ByteLevel - estilo GPT-2**

In [31]:
tok_byte = Tokenizer(models.BPE())
tok_byte.pre_tokenizer = pre_tokenizers.ByteLevel()
print(tok_byte.pre_tokenizer.pre_tokenize_str(frase))

[('ĠNÃ£o', (0, 3)), (',', (3, 4)), ('ĠserÃ¡', (4, 9)), ('Ġpunido', (9, 16)), ('Ġo', (16, 18)), ('Ġcriminoso', (18, 28)), ('.', (28, 29))]


# Metaspace (SentencePiece style)

In [32]:
tok_meta = Tokenizer(models.BPE())
tok_meta.pre_tokenizer = pre_tokenizers.Metaspace()
print(tok_meta.pre_tokenizer.pre_tokenize_str(frase))

[('▁Não,', (0, 4)), ('▁será', (4, 9)), ('▁punido', (9, 16)), ('▁o', (16, 18)), ('▁criminoso.', (18, 29))]


#Treinamento


In [33]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# 1. Criar o tokenizador
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=30000,  # ajuste conforme o corpus
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"]
)

# 2. Gerador que lê o arquivo linha por linha
def ler_corpus_txt(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                yield linha

# 3. Treinamento
tokenizer.train_from_iterator(
    ler_corpus_txt("sinopses_sem_titulos.txt"),
    trainer=trainer
)

# 4. Salvar
tokenizer.save("bpe_tokenizer.json")


# Encode

In [38]:
# 03_tokenizer_encode.ipynb
# Pipeline de tokenização: normalização → pré-tokenização → modelo → pós-processamento

from tokenizers import Tokenizer, normalizers, pre_tokenizers, processors
from tokenizers.normalizers import NFD, StripAccents, Lowercase
from tokenizers.pre_tokenizers import Whitespace, Digits, Sequence
from tokenizers.processors import TemplateProcessing

print("### Pipeline de tokenização ###")
print(" Normalization")
print(" Pre-tokenization")
print(" Model")
print(" Post-processing\n")

# Carregar o tokenizador treinado (BPE)
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

# -----------------------------------------------------------
# Normalization
# -----------------------------------------------------------
print("# Normalization")
normalizer = normalizers.Sequence([
    NFD(),          # decomposição de acentos
    Lowercase(),    # tudo minúsculo
    StripAccents()  # remove acentos
])
texto = "Héllò hôw are ü?"
print("Antes:", texto)
print("Depois:", normalizer.normalize_str(texto), "\n")
tokenizer.normalizer = normalizer

# -----------------------------------------------------------
# Pre-tokenization
# -----------------------------------------------------------
print("# Pre-tokenization")
pre_tok = Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation(),
    Digits(individual_digits=True)
])
texto2 = "Hello! How are you? Tenho R$ 213,12."
print("Pré-tokenização:", pre_tok.pre_tokenize_str(texto2), "\n")
tokenizer.pre_tokenizer = pre_tok

# -----------------------------------------------------------
# Model
# -----------------------------------------------------------
print("# Model: BPE (Byte Pair Encoding)")
# já carregado do arquivo bpe_tokenizer.json

# -----------------------------------------------------------
# Post-processing
# -----------------------------------------------------------
print("# Post-processing (TemplateProcessing)")
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", 1), ("[SEP]", 2)],
)

# -----------------------------------------------------------
# Aplicando tudo
# -----------------------------------------------------------
encoded = tokenizer.encode("olá mundo")
print("Tokens IDs:", encoded.ids)
print("Tokens:", encoded.tokens)


### Pipeline de tokenização ###
 Normalization
 Pre-tokenization
 Model
 Post-processing

# Normalization
Antes: Héllò hôw are ü?
Depois: hello how are u? 

# Pre-tokenization
Pré-tokenização: [('Hello', (0, 5)), ('!', (5, 6)), ('How', (7, 10)), ('are', (11, 14)), ('you', (15, 18)), ('?', (18, 19)), ('Tenho', (20, 25)), ('R', (26, 27)), ('$', (27, 28)), ('2', (29, 30)), ('1', (30, 31)), ('3', (31, 32)), (',', (32, 33)), ('1', (33, 34)), ('2', (34, 35)), ('.', (35, 36))] 

# Model: BPE (Byte Pair Encoding)
# Post-processing (TemplateProcessing)
Tokens IDs: [1, 45, 103, 214, 2]
Tokens: ['[CLS]', 'o', 'la', 'mundo', '[SEP]']


## Bytelevel vs SentencePiece

In [35]:
# 03_bytelevel_vs_sentencepiece.ipynb
# Comparando ByteLevel (GPT-2) vs SentencePiece (mT5)

from transformers import AutoTokenizer
import unicodedata

# -----------------------------
# 1️⃣ Modelos
# -----------------------------
BYTELEVEL_MODEL = "openai-community/gpt2"
SENTPIECE_MODEL = "google/mt5-small"

tok_byte = AutoTokenizer.from_pretrained(BYTELEVEL_MODEL)
tok_spm  = AutoTokenizer.from_pretrained(SENTPIECE_MODEL)

# Garantir pad_token
if tok_byte.pad_token is None and hasattr(tok_byte, "eos_token"):
    tok_byte.pad_token = tok_byte.eos_token

# -----------------------------
# 2️⃣ Texto de exemplo
# -----------------------------
text = "Vamos comer, vovó! 🙂"
print(f"Texto: {text}\n")

# -----------------------------
# 3️⃣ Tokenização
# -----------------------------
def encode_details(tokenizer, name):
    enc = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    print(f"=== {name} ===")
    print("Tokens:", tokens)
    print("IDs:", ids)
    print("Qtd tokens:", len(tokens))
    print("Decoded:", tokenizer.decode(ids))
    print("Offsets:", offsets)
    print()

encode_details(tok_byte, "ByteLevel (GPT-2)")
encode_details(tok_spm, "SentencePiece (mT5)")

# -----------------------------
# 4️⃣ Comparação Unicode (opcional)
# -----------------------------
def show_unicode_chars(s):
    for ch in s:
        name = unicodedata.name(ch, "UNKNOWN")
        print(f"{repr(ch)} -> {name}")

print("\nCaracteres Unicode do texto:")
show_unicode_chars(text)


Could not extract SentencePiece model from C:\Users\Dell\.cache\huggingface\hub\models--google--mt5-small\snapshots\73fb5dbe4756edadc8fbe8c769b0a109493acf7a\spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in C:\Users\Dell\.cache\huggingface\hub\models--google--mt5-small\snapshots\73fb5dbe4756edadc8fbe8c769b0a109493acf7a\spiece.model

# Avaliação

In [ ]:
from tokenizers import Tokenizer
import numpy as np

# Carrega o tokenizador treinado
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

# Corpus de teste (pode ser parte do seu corpus real)
test_texts = [
    "O rato roeu a roupa do rei de Roma.",
    "Aprender tokenização é divertido!",
    "GPT-2 e mT5 usam abordagens diferentes.",
    "Python é ótimo para NLP 😄",
]

# Funções auxiliares
def count_chars(text):
    return len(text)

def count_words(text):
    return len(text.split())

def evaluate_tokenizer(tokenizer, texts):
    stats = []
    for t in texts:
        enc = tokenizer.encode(t)
        stats.append({
            "text": t,
            "chars": count_chars(t),
            "words": count_words(t),
            "tokens": len(enc.tokens),
            "unk": enc.tokens.count("<unk>"),
            "decoded_ok": (tokenizer.decode(enc.ids) == t)
        })
    return stats

stats = evaluate_tokenizer(tokenizer, test_texts)

# Converter para métricas agregadas
import pandas as pd
df = pd.DataFrame(stats)

tpc = (df["tokens"] / df["chars"]).mean()
tpw = (df["tokens"] / df["words"]).mean()
unk_rate = (df["unk"].sum() / df["tokens"].sum()) * 100
decode_acc = (df["decoded_ok"].mean()) * 100


print("algo deveria estar aqui em baixo")
print(tokenizer.token_to_id("<pad>"))
print("=== Métricas de eficiência ===")
print(f"Tokens por caractere (TPC): {tpc:.3f}")
print(f"Tokens por palavra (TPW): {tpw:.3f}")
print(f"Percentual de <unk>: {unk_rate:.2f}%")
print(f"Reversibilidade (decode == original): {decode_acc:.1f}%")
print(f"Tamanho médio da sequência: {df['tokens'].mean():.1f} tokens/frase")


algo deveria estar aqui em baixo
None
=== Métricas de eficiência ===
Tokens por caractere (TPC): 0.490
Tokens por palavra (TPW): 2.743
Percentual de <unk>: 6.25%
Reversibilidade (decode == original): 0.0%
Tamanho médio da sequência: 16.0 tokens/frase
